# Module 06: Geolocation EDA & Cleaning

### 1. Load Raw Data (Bronze Layer)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data")
RAW_DATA_PATH = DATA_DIR / "olist_geolocation_dataset.csv"

# Ensure the file exists before loading
assert RAW_DATA_PATH.exists(), f"Source file not found at: {RAW_DATA_PATH.resolve()}"

geolocation = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded raw dataset shape: {geolocation.shape}")
geolocation.head(3)

Loaded raw dataset shape: (1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


### 2. Data Audit & Missing Values Check
Analyzing data types, missing value distribution, and column cardinality.

In [2]:
audit_df = pd.DataFrame(
    {
        "Data Type": geolocation.dtypes,
        "Missing Values": geolocation.isna().sum(),
        "Missing %": (geolocation.isna().mean() * 100).round(2),
        "Unique Values": geolocation.nunique(),
    }
)
display(audit_df)

,Data Type,Missing Values,Missing %,Unique Values
geolocation_zip_code_prefix,int64,0,0.0,19015
geolocation_lat,float64,0,0.0,717360
geolocation_lng,float64,0,0.0,717613
geolocation_city,object,0,0.0,8011
geolocation_state,object,0,0.0,27


### 3. Text Standardization & Silver Layer Initialization

To eliminate discrepancies caused by formatting inconsistencies, we standardize all text fields:
- **String Cleaning:** Strip unwanted whitespaces and enforce proper capitalization for city names (`.title()`).
- **State Code Harmonization:** Ensure all state abbreviations conform strictly to uppercase ISO standards.
- **ZIP Code Formatting:** Convert postal code prefixes into 5-digit zero-padded strings to prepare for accurate spatial and geographic operations.

In [3]:
geolocation_clean = geolocation.copy()

placeholders = ["", "-", "n/a", "null", "undefined", "unknown"]

geolocation_clean["geolocation_city"] = geolocation_clean["geolocation_city"].replace(placeholders, np.nan)
geolocation_clean["geolocation_city"] = geolocation_clean["geolocation_city"].str.split().str.join(" ").str.title()
geolocation_clean["geolocation_state"] = geolocation_clean["geolocation_state"].astype(str).str.strip().str.upper()
geolocation_clean["geolocation_zip_code_prefix"] = geolocation_clean["geolocation_zip_code_prefix"].astype(str).str.zfill(5)

display(geolocation_clean.head(3))

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,Sao Paulo,SP
1,01046,-23.546081,-46.644820,Sao Paulo,SP
2,01046,-23.546129,-46.642951,Sao Paulo,SP


In [4]:
geolocation_clean.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

### 4. Data Type Optimization

To transition our dataset from the Bronze layer to the Silver layer, we enforce robust data types:
- **Zip Code Prefix (`string` with `zfill(5)`):** Preserves 5-digit formatting with zero-padding and enforces explicit string type to guarantee exact matching during geographic joins.
- **Coordinates (`float64`):** Ensures physical latitude and longitude values retain high-precision numeric types.
- **City & State (`string` / `category`):** Standardizes text fields and casts state codes to categorical types to optimize memory overhead and improve filtering speed.

In [5]:
geolocation_clean = geolocation.copy()

geolocation_clean["geolocation_zip_code_prefix"] = geolocation_clean["geolocation_zip_code_prefix"].astype(str).str.zfill(5).astype("string")
geolocation_clean["geolocation_lat"] = geolocation_clean["geolocation_lat"].astype("float64")
geolocation_clean["geolocation_lng"] = geolocation_clean["geolocation_lng"].astype("float64")
geolocation_clean["geolocation_city"] = geolocation_clean["geolocation_city"].astype("string")
geolocation_clean["geolocation_state"] = geolocation_clean["geolocation_state"].astype("category")

display(geolocation_clean.dtypes.to_frame(name="Silver Layer Data Type"))

,Silver Layer Data Type
geolocation_zip_code_prefix,string[python]
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,string[python]
geolocation_state,category


### 5. Data Integrity & Anomaly Audit

To ensure data quality prior to finalizing the Silver layer, we execute comprehensive validation checks:
- **Geographic Bounds:** Screening for latitude and longitude values falling outside Brazil's bounding box (`geolocation_lat`, `geolocation_lng`).
- **Zero Coordinates:** Checking for technical artifacts with `(0, 0)` coordinates.
- **State Code Validation:** Verifying that all state abbreviations strictly conform to the 2-character ISO format (`geolocation_state`).
- **Text Hygiene:** Inspecting city names (`geolocation_city`) for missing values (`NaN`) or empty strings.
- **Redundancy Analysis:** Identifying exact duplicate rows across all fields.
- **Format Integrity:** Confirming that all 5-digit postal code prefixes contain strictly numeric characters.
- **Geographic Consistency:** Detecting ZIP code prefixes that are ambiguously mapped to multiple states.

In [6]:
anomaly_checks = {
    "Out-of-bounds coordinates": (
        (geolocation_clean["geolocation_lat"] > 6.0) | (geolocation_clean["geolocation_lat"] < -34.0) |
        (geolocation_clean["geolocation_lng"] > -34.0) | (geolocation_clean["geolocation_lng"] < -74.0)
    ),
    "Zero coordinates (0, 0)": (
        (geolocation_clean["geolocation_lat"] == 0) & (geolocation_clean["geolocation_lng"] == 0)
    ),
    "Invalid state code length (!= 2 chars)": (
        ~geolocation_clean["geolocation_state"].astype(str).str.len().eq(2)
    ),
    "Empty city names": (
        geolocation_clean["geolocation_city"].isna() | (geolocation_clean["geolocation_city"].str.strip() == "")
    ),
    "Exact duplicate rows": (
        geolocation_clean.duplicated()
    ),
    "Non-numeric ZIP code prefixes": (
        ~geolocation_clean["geolocation_zip_code_prefix"].str.isdigit()
    ),
    "ZIP codes mapped to multiple states": (
        geolocation_clean.groupby("geolocation_zip_code_prefix")["geolocation_state"].transform("nunique") > 1
    )
}

for label, condition in anomaly_checks.items():
    print(f"{label}: {condition.sum():,}")

Out-of-bounds coordinates: 42
Zero coordinates (0, 0): 0
Invalid state code length (!= 2 chars): 0
Empty city names: 0
Exact duplicate rows: 261,831
Non-numeric ZIP code prefixes: 0
ZIP codes mapped to multiple states: 865


#### Redundancy and Out-of-Bounds Filtering
Purging exact duplicate rows and invalid coordinate records falling outside Brazil's geographic boundaries to establish a clean foundation for the Silver layer.

In [7]:
# 1. Remove exact duplicate rows
initial_len = len(geolocation_clean)
geolocation_clean = geolocation_clean.drop_duplicates()
print(f"Dropped duplicate rows: {initial_len - len(geolocation_clean):,}")

# 2. Filter out-of-bounds coordinates based on Brazil's geographic boundaries
lat_out_of_bounds = (geolocation_clean["geolocation_lat"] > 6.0) | (geolocation_clean["geolocation_lat"] < -34.0)
lng_out_of_bounds = (geolocation_clean["geolocation_lng"] > -34.0) | (geolocation_clean["geolocation_lng"] < -74.0)
out_of_bounds_mask = lat_out_of_bounds | lng_out_of_bounds

print(f"Dropped out-of-bounds coordinate rows: {out_of_bounds_mask.sum():,}")
geolocation_clean = geolocation_clean[~out_of_bounds_mask]

# 3. Filter out ZIP codes mapped to multiple conflicting states
multi_state_mask = geolocation_clean.groupby("geolocation_zip_code_prefix")["geolocation_state"].transform("nunique") > 1
print(f"Dropped rows with conflicting states across ZIP codes: {multi_state_mask.sum():,}")
geolocation_clean = geolocation_clean[~multi_state_mask]

# 4. Summary of the cleaned dataset before spatial aggregation
print(f"Final Silver Layer shape (pre-aggregation): {geolocation_clean.shape}")

Dropped duplicate rows: 261,831
Dropped out-of-bounds coordinate rows: 33
Dropped rows with conflicting states across ZIP codes: 572
Final Silver Layer shape (pre-aggregation): (737727, 5)


### 6. Silver Layer Validation & Export
Comparing Bronze vs. Silver metrics (shape, types, and record count) and persisting the dataset to Parquet.

In [8]:
print("Pipeline Summary: Bronze vs Silver Layer")
print(f"Original shape (Bronze): {geolocation.shape}")
print(f"Cleaned shape (Silver): {geolocation_clean.shape}")
print(f"Total rows dropped: {len(geolocation) - len(geolocation_clean):,}")

dtypes_comparison = pd.DataFrame({
    'Bronze (Raw)': geolocation.dtypes,
    'Silver (Clean)': geolocation_clean.dtypes
})

print("\nData Types Transformation:")
display(dtypes_comparison)

Pipeline Summary: Bronze vs Silver Layer
Original shape (Bronze): (1000163, 5)
Cleaned shape (Silver): (737727, 5)
Total rows dropped: 262,436

Data Types Transformation:


,Bronze (Raw),Silver (Clean)
geolocation_zip_code_prefix,int64,string[python]
geolocation_lat,float64,float64
geolocation_lng,float64,float64
geolocation_city,object,string[python]
geolocation_state,object,category


## 7. Data Quality Assertions & Export

Before persisting the clean dataset to the Silver layer, we execute automated data quality checks tailored to spatial and geographic integrity:

* **Geographic Bounds Validity:** Confirm all latitude and longitude values strictly adhere to Brazil's geographic bounding box.
* **Mandatory Attributes:** Verify core fields (`geolocation_zip_code_prefix`, `geolocation_lat`, `geolocation_lng`, `geolocation_city`, `geolocation_state`) contain no missing values.
* **State Code Formatting:** Ensure all state abbreviations conform strictly to the 2-character ISO format.
* **Dataset Validity:** Ensure the resulting dataset is non-empty before writing to storage.

In [9]:
lat_valid = (geolocation_clean['geolocation_lat'] >= -34.0) & (geolocation_clean['geolocation_lat'] <= 6.0)
lng_valid = (geolocation_clean['geolocation_lng'] >= -74.0) & (geolocation_clean['geolocation_lng'] <= -34.0)
assert (lat_valid & lng_valid).all(), "Assertion failed: Out-of-bounds coordinates found"

critical_cols = ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
for col in critical_cols:
    assert geolocation_clean[col].notna().all(), f"Assertion failed: NULL values found in mandatory column '{col}'"

invalid_states = ~geolocation_clean['geolocation_state'].astype(str).str.len().eq(2)
assert not invalid_states.any(), "Assertion failed: Invalid state code length found"

assert len(geolocation_clean) > 0, "Assertion failed: Cleaned dataset is empty"

print("All quality checks passed successfully. Saving to Silver layer.")
geolocation_clean.to_parquet("../data/geolocation_clean.parquet", index=False)
print("Geolocation dataset successfully saved to Parquet.")

All quality checks passed successfully. Saving to Silver layer.
Geolocation dataset successfully saved to Parquet.
